# Blog Post Refiner | Prompt Chaining

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Define shared state
class ChainState(TypedDict):
    input: str
    draft: str
    analysis: str
    final_output: str

In [5]:
# Step 1: Generate a draft
def generate_draft(state: ChainState) -> dict:
    response = model.invoke(
        f"Write a short blog post about: {state['input']}"
    )
    return {"draft": response.content}

In [6]:
# Step 2: Analyze the draft — extract specific, actionable issues
# Mark showstoppers as [CRITICAL] so downstream steps can gate on them.
def analyze_draft(state: ChainState) -> dict:
    response = model.invoke(
        f"Analyze this blog post and return a structured list of specific issues.\n"
        f"For each issue, state the exact problem and location.\n"
        f"Format: '- [CATEGORY] issue description' where CATEGORY is one of: "
        f"CRITICAL, FACTUAL, GRAMMAR, CLARITY, STRUCTURE, TONE.\n"
        f"Use CRITICAL for factual errors, misleading claims, or logical contradictions.\n"
        f"Example: '- [CRITICAL] Paragraph 2 claims X but this is factually wrong'\n\n"
        f"{state['draft']}"
    )
    # Validation gate: flag if CRITICAL issues exist so next step prioritizes them
    analysis = response.content
    has_critical = "[CRITICAL]" in analysis
    if has_critical:
        print(f"⚠ Validation gate: CRITICAL issues detected — downstream step will prioritize fixes.")
    return {"analysis": analysis}

In [7]:
# Step 3: Produce final version — explicitly addresses each issue from Step 2
# If the analysis flagged CRITICAL issues, those gate the rewrite priority.
def finalize(state: ChainState) -> dict:
    response = model.invoke(
        f"Rewrite this blog post by addressing EACH specific issue from the analysis below.\n"
        f"If the analysis flagged CRITICAL issues, prioritize fixing those first.\n"
        f"For every issue listed, make the corresponding fix in the draft.\n\n"
        f"Draft:\n{state['draft']}\n\n"
        f"Issues to address:\n{state['analysis']}"
    )
    return {"final_output": response.content}

In [8]:
# Build the chain (add_sequence wires nodes in order automatically)
graph = StateGraph(ChainState)
graph.add_sequence([("generate", generate_draft), ("analyze", analyze_draft), ("finalize", finalize)])
graph.add_edge(START, "generate")
graph.add_edge("finalize", END)

chain = graph.compile()

In [9]:
# Plot the workflow
plot_mermaid(chain)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	analyze(analyze)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	analyze --> finalize;
	generate --> analyze;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
result = chain.invoke({"input": "AI agents in production"})
print(result["final_output"])

⚠ Validation gate: CRITICAL issues detected — downstream step will prioritize fixes.

**Title: Embracing the Future: AI Agents Transforming Production**

In an era where technology is advancing at a breathtaking pace, AI agents have evolved from early implementations to becoming critical components in the production landscape. For decades, industries worldwide have been increasingly leveraging AI to enhance efficiency, innovation, and competitiveness. Let’s delve into how AI agents are revolutionizing production and what the future may hold.

**Revolutionizing Efficiency and Precision**

One of the most significant advantages of integrating AI agents into production is the unprecedented increase in efficiency and precision for specific tasks. AI systems can efficiently handle complex analytical functions with remarkable speed and accuracy, such as real-time data analysis to predict maintenance needs and optimize resource allocation. While these systems greatly enhance operational streamlining, human oversight is still essential in many areas, ensuring that their proacti

In [11]:
# Streaming

stream_invoke(chain, {"input": "AI agents in production"})

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

⚠ Validation gate: CRITICAL issues detected — downstream step will prioritize fixes.

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'input': 'AI agents in production',
 'draft': '**Title: Embracing the Future: AI Agents in Production**\n\nIn an era where technology is advancing at a breathtaking pace, AI agents have transitioned from being mere possibilities to pivotal components in the production landscape. Industries worldwide are increasingly leveraging AI to enhance efficiency, innovation, and competitiveness. Let’s delve into how AI agents are revolutionizing production and what the future may hold.\n\n**Revolutionizing Efficiency and Precision**\n\nOne of the most significant advantages of integrating AI agents into production is the unparalleled increase in efficiency and precision. AI systems are designed to handle complex tasks with remarkable speed and accuracy, far surpassing the capabilities of manual labor. These agents can analyze vast amounts of data in real-time, enabling them to predict maintenance needs, optimize resource allocation, and streamline operations. This proactive approach minimizes do